In [ ]:
# Copyright 2024 Google LLC
#
# Licensed under the Apache License, Version 2.0 (the "License");
# you may not use this file except in compliance with the License.
# You may obtain a copy of the License at
#
#     https://www.apache.org/licenses/LICENSE-2.0
#
# Unless required by applicable law or agreed to in writing, software
# distributed under the License is distributed on an "AS IS" BASIS,
# WITHOUT WARRANTIES OR CONDITIONS OF ANY KIND, either express or implied.
# See the License for the specific language governing permissions and
# limitations under the License.

# Multi-Agent Self-Correcting Workflow

<table align="left">
  <td style="text-align: center">
    <a href="https://colab.research.google.com/github/GoogleCloudPlatform/generative-ai/blob/main/gemini/orchestration/langgraph/langgraph_fact_checker.ipynb">
      <img src="https://cloud.google.com/ml-engine/images/colab-logo-32px.png" alt="Google Colaboratory logo"><br> Run in Colab
    </a>
  </td>
  <td style="text-align: center">
    <a href="https://github.com/GoogleCloudPlatform/generative-ai/blob/main/gemini/orchestration/langgraph/langgraph_fact_checker.ipynb">
      <img src="https://cloud.google.com/ml-engine/images/github-logo-32px.png" alt="GitHub logo"><br> View on GitHub
    </a>
  </td>
  <td style="text-align: center">
    <a href="https://console.cloud.google.com/vertex-ai/workbench/deploy-notebook?download_url=https://raw.githubusercontent.com/GoogleCloudPlatform/generative-ai/main/gemini/orchestration/langgraph/langgraph_fact_checker.ipynb">
      <img src="https://lh3.googleusercontent.com/UiNoO41c1N1s8WGNy1t23hwQiGzvZaS0F4mYhGttX91hP2eHk6O-8C1j-I3u1h-uHwGjL693aXjH-t5aU7Z4b3g3zZ7YjZ5m0L3l" alt="Vertex AI logo"><br> Open in Vertex AI Workbench
    </a>
  </td>
</table>
<br><br><br>

## Overview

While basic multi-agent systems often suffer from hallucination and context loss, production enterprise systems require strict quality control. This notebook demonstrates how to build an **Enterprise Editorial Assembly Line** using [LangGraph](https://python.langchain.com/docs/langgraph) and Gemini on Google Cloud.

Instead of relying on a single "Omni-Agent" or a highly autonomous (and unpredictable) supervisor, this sample showcases a **Self-Correcting Cyclic Graph** where specialized agents operate in a predictable sequence:

1. **Extractor:** Isolates distinct factual claims from an input document.
2. **Researcher (Agentic RAG):** Natively uses tools to fetch ground-truth data for each claim.
3. **Editor:** Rewrites the document using only verified facts.
4. **Reviewer (Critic):** Evaluates the final draft against the ground-truth notes.

A deterministic routing loop forces the Editor to rewrite the content if the Reviewer agent rejects the draft, ensuring high quality before human review.

## Setup

First, install the required packages.

In [ ]:
!pip install -qU langgraph langchain-google-genai pydantic

### Authentication

Configure your Gemini API key. If you are running this in Colab, you can use the built-in secrets manager, or input it manually below.

In [ ]:
import os
import getpass

if "GOOGLE_API_KEY" not in os.environ:
    os.environ["GOOGLE_API_KEY"] = getpass.getpass("Enter your Google Gemini API Key: ")

## 1. Initialize Gemini and Tools

We initialize Gemini Flash Latest. We then define a native tool `query_internal_wiki` which acts as our mock internal ground-truth database. We bind this tool specifically to our Researcher agent persona.

In [ ]:
from langchain_google_genai import ChatGoogleGenerativeAI
from langchain_core.tools import tool

# Initialize Gemini 1.5 Flash (using -latest to ensure highest compatibility)
llm = ChatGoogleGenerativeAI(model="gemini-flash-latest", temperature=0)

@tool
def query_internal_wiki(query: str) -> str:
    """Queries the internal engineering wiki for verified corporate facts, policies, and limits."""
    print(f"   [Tool Executed natively] query='{query}'")
    query = query.lower()

    # Mocking a robust internal knowledge base
    if "deprecation" in query or "app engine" in query:
        return "UPDATE (Q3): App Engine standard environment deprecation date has been officially extended to January 15th, 2026."
    elif "timeout" in query or "cloud run" in query:
        return "DOCS: Cloud Run maximum request timeout was increased to 60 minutes in the v2 API update."
    elif "rate limit" in query or "gateway" in query:
        return "POLICY: Internal API Gateway rate limit is now 500 requests/min for authenticated users."

    return "No verified information found in the wiki."

# Bind the tool specifically to the researcher persona
researcher_llm = llm.bind_tools([query_internal_wiki])

## 2. Define the Agent State (`TypedDict`)

In LangGraph, the state acts as the shared whiteboard between agents. By strongly typing this state, we ensure that the Editor only reads the Researcher's notes, preventing context collapse during handoffs.

In [ ]:
from typing import TypedDict, Annotated, Sequence
import operator
from langchain_core.messages import BaseMessage

class AgentState(TypedDict):
    original_document: str
    extracted_claims: list[str]
    research_notes: list[str]
    draft_revision: str
    reviewer_feedback: str
    approved: bool
    # LangGraph requires a messages key to maintain graph execution history
    messages: Annotated[Sequence[BaseMessage], operator.add]

## 3. Define the Specialized Agent Nodes

Instead of one generic agent, we define four strict personas. Notice how the `researcher_node` actively checks for `response.tool_calls` and executes the Python function if the LLM requests it.

In [ ]:
import json
from langchain_core.messages import HumanMessage, SystemMessage

def _get_text(response):
    """Safely extracts text from the LLM response (handles both string and block lists)."""
    if isinstance(response.content, list):
        return "".join([block.get("text", "") for block in response.content if isinstance(block, dict)])
    return str(response.content)

def extractor_node(state: AgentState):
    print("▶ [Extractor] Finding claims...")
    prompt = f"Extract fact-checkable claims from this text. Output ONLY a valid JSON array of strings. Do not output any markdown formatting. Text: {state['original_document']}"
    response = llm.invoke([SystemMessage(content="You are the Extractor."), HumanMessage(content=prompt)])
    content_str = _get_text(response).replace("```json", "").replace("```", "").strip()
    try:
        claims = json.loads(content_str)
        if not isinstance(claims, list): claims = [str(claims)]
    except:
        claims = [content_str]
    return {"extracted_claims": claims}

def researcher_node(state: AgentState):
    print("▶ [Researcher] Fact-checking (Agentic RAG)...")
    notes = []
    for claim in state.get('extracted_claims', []):
        if not claim.strip(): continue
        prompt = f"You MUST use your tool 'query_internal_wiki' to fact-check this claim. Do NOT answer from your own knowledge. Claim: '{claim}'"
        response = researcher_llm.invoke([HumanMessage(content=prompt)])

        # Execute tool if the LLM invoked it
        if response.tool_calls:
            for tool_call in response.tool_calls:
                if tool_call['name'] == 'query_internal_wiki':
                    tool_result = query_internal_wiki.invoke(tool_call['args'])
                    notes.append(f"[Claim]: {claim} -> [Wiki Fact]: {tool_result}")
        else:
            notes.append(f"[Claim]: {claim} -> [LLM Note]: {_get_text(response)}")

    return {"research_notes": notes}

def editor_node(state: AgentState):
    print("▶ [Editor] Drafting Revision...")
    feedback = state.get('reviewer_feedback', 'None')
    prompt = f"Original Document: {state['original_document']}\\n\\nResearcher's Ground Truth Notes: {state['research_notes']}\\n\\nPrevious Reviewer Feedback: {feedback}\\n\\nTask: Rewrite the original document using ONLY the verified facts from the Researcher's notes. Do not hallucinate."
    response = llm.invoke([SystemMessage(content="You are the strict Editor."), HumanMessage(content=prompt)])
    return {"draft_revision": _get_text(response)}

def reviewer_node(state: AgentState):
    print("▶ [Reviewer] Auditing draft against ground truth...")
    prompt = f"Draft: '{state['draft_revision']}'\\nNotes: '{state['research_notes']}'\\nDoes the draft accurately reflect the notes and ONLY the notes? Output JSON with 'approved': true/false and 'feedback'."
    response = llm.invoke([SystemMessage(content="You are a strict Reviewer. Output JSON."), HumanMessage(content=prompt)])
    content_str = _get_text(response)
    try:
        review = json.loads(content_str.replace("```json", "").replace("```", "").strip())
        return {"reviewer_feedback": review.get("feedback", ""), "approved": review.get("approved", False)}
    except:
        return {"reviewer_feedback": "Failed to parse JSON", "approved": False}

## 4. The Deterministic Router (Self-Correction)

This function reads the boolean output from the Reviewer. If the draft is rejected, it forces the graph back to the `editor` node, creating our self-correcting loop.

In [ ]:
from langgraph.graph import END

def reviewer_router(state: AgentState):
    if state.get("approved", False):
        print("   ✅ Approved! Routing to END.")
        return END
    else:
        print(f"   ❌ Rejected! Routing back to Editor. Feedback: {state['reviewer_feedback']}")
        return "editor"

## 5. Build and Compile the Graph

We wire the nodes together into a `StateGraph`, defining the strictly sequential edges, and appending our conditional cyclic edge at the end.

In [ ]:
from langgraph.graph import StateGraph

workflow = StateGraph(AgentState)

# Add Nodes
workflow.add_node("extractor", extractor_node)
workflow.add_node("researcher", researcher_node)
workflow.add_node("editor", editor_node)
workflow.add_node("reviewer", reviewer_node)

# Define Linear Edges
workflow.set_entry_point("extractor")
workflow.add_edge("extractor", "researcher")
workflow.add_edge("researcher", "editor")
workflow.add_edge("editor", "reviewer")

# Define Conditional Edge (The self-correcting loop)
workflow.add_conditional_edges("reviewer", reviewer_router)

# Compile the workflow into a runnable application
app = workflow.compile()

## 6. Execute the Swarm

We pass in a piece of outdated, legacy documentation. Watch as the agents autonomously extract the claims, fetch the 2026 data, rewrite the document, and review it for accuracy.

In [ ]:
# A realistic piece of legacy documentation containing multiple outdated facts
outdated_documentation = """
# Cloud Platform Migration Guide

Please be aware that the legacy App Engine standard environment will be fully deprecated on October 1st, 2025. All services must migrate to Cloud Run before this deadline.

When migrating, keep in mind that the maximum timeout for Cloud Run requests is strictly capped at 15 minutes.

Additionally, our internal API gateway enforces a rate limit of 100 requests per minute per user. Please batch your database queries accordingly.
"""

initial_state = {"original_document": outdated_documentation}

print("Starting the swarm...\\n")
final_state = app.invoke(initial_state)

print("\\n" + "="*50)
print("FINAL APPROVED DOCUMENT")
print("="*50)
print(final_state['draft_revision'])